# DR model v2: fine-tune, calibrate on validation, measure once on a held-out test split

**Why v2.** The v1 model (frozen MobileNetV2 backbone, 8 epochs, nearest-neighbour resize, untuned threshold 0.5)
reached sensitivity 0.542 on its validation fold, below the proposal's 85% target. This notebook follows the
proposal's method (section 3.3: MobileNetV2 transfer learning, fine-tuned) more fully:

1. **Split by image group** (augmented copies of one photo stay together): 70% train / 15% validation / 15% test.
2. **Fine-tune**: train the head with the backbone frozen, then unfreeze the top MobileNetV2 blocks at a low
   learning rate. Bilinear (anti-aliased) resize keeps small lesions that nearest-neighbour resizing drops.
3. **Choose the threshold on validation only**: the highest threshold with sensitivity >= 0.85 (same rule as
   the app's `select_threshold`, positive = score >= threshold).
4. **Measure the test split once** at that threshold, with 95% bootstrap intervals, plus results per ICDR grade
   and for *referable* DR (grade >= 2, the definition used by the Gulshan et al. 2016 baseline).
5. **Export** `model.tflite` (float32) and check it agrees with the Keras model.

### Run it (about 1.5-2.5 hours, no need to watch)
1. Kaggle: **Create -> New Notebook -> File -> Import Notebook** -> choose this file.
2. Right panel -> **Session options**: **Accelerator: GPU T4 x2**, **Internet: On**.
3. **Add Input** -> search `eyepacs-aptos-messidor-diabetic-retinopathy` (by *ascanipek*) -> **Add**.
4. Top right **Save Version** -> **Save & Run All (Commit)** -> Save. It runs in the background; you can close the tab.
5. When the version finishes: open it -> **Output** -> download **`dr_v2_results.zip`** and send it to Claude Code.
   Also screenshot the final **CHECKS** cell.

Nothing here is typed by hand: every number in `results.json` is computed from the predictions this run makes.

## 1. Settings

In [ ]:
import glob, hashlib, json, os, re, shutil, time, zipfile

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
T0 = time.time()

# Smoke-test mode runs every cell on a tiny fake dataset (used by the developer before spending GPU time).
SMOKE = os.environ.get("DR_SMOKE_TEST") == "1"
INPUT_ROOT = os.environ.get("DR_INPUT_ROOT", "/kaggle/input")
OUT = os.environ.get("DR_OUT_ROOT", "/kaggle/working")

IMG = 224                      # the app's input size
BATCH = 8 if SMOKE else 64
STEPS_PER_EPOCH = None if SMOKE else 1500   # ~96k images per epoch; the full train split is seen over several epochs
HEAD_EPOCHS = 1 if SMOKE else 2
FINE_EPOCHS = 1 if SMOKE else 8
FINE_TUNE_FROM = "block_13_expand"         # unfreeze MobileNetV2 from this layer to the top
VAL_MONITOR_N = 64 if SMOKE else 6000      # validation subset checked after every epoch (full split used later)
TIME_BUDGET_MIN = 150                      # stop fine-tuning after this many minutes of training
TARGET_SENSITIVITY = 0.85                  # proposal targets
TARGET_SPECIFICITY = 0.80
BOOTSTRAP = 200 if SMOKE else 1000
IMG_EXTS = (".png", ".jpg", ".jpeg")

GPUS = tf.config.list_physical_devices("GPU")
if GPUS and not SMOKE:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("TensorFlow", tf.__version__, "| GPUs:", len(GPUS), "| smoke test:", SMOKE)

## 2. Find the dataset and build the image table

In [ ]:
def make_smoke_dataset(root, per_class=24):
    # Fake fundus-like images whose brightness depends on the grade, so the code path can learn something.
    from PIL import Image
    rng = np.random.default_rng(SEED)
    for grade in range(5):
        os.makedirs(os.path.join(root, "train", str(grade)), exist_ok=True)
        for i in range(per_class):
            base = 60 + grade * 35
            arr = np.clip(rng.normal(base, 25, (300, 300, 3)), 0, 255).astype(np.uint8)
            Image.fromarray(arr).save(os.path.join(root, "train", str(grade), f"img{grade}_{i}.png"))
            if i % 3 == 0:  # an augmented copy that must stay in the same split as its original
                Image.fromarray(arr[:, ::-1]).save(os.path.join(root, "train", str(grade), f"img{grade}_{i}_aug1.png"))
    return root

if SMOKE:
    DATASET_PATH = make_smoke_dataset(os.path.join(OUT, "smoke_dataset"))
else:
    hits = []
    for pattern in ("*eyepacs-aptos-messidor*", "*/*eyepacs-aptos-messidor*", "*/*/*eyepacs-aptos-messidor*"):
        hits += glob.glob(os.path.join(INPUT_ROOT, pattern))
    assert hits, "Dataset not attached: Add Input -> search 'eyepacs-aptos-messidor-diabetic-retinopathy' -> Add."
    DATASET_PATH = sorted(hits)[0]
print("dataset:", DATASET_PATH)

# One walk over the folder (much faster than one glob per CSV row).
all_files = [os.path.join(r, f) for r, _, fs in os.walk(DATASET_PATH) for f in fs]
images = [f for f in all_files if f.lower().endswith(IMG_EXTS)]
csv_files = [f for f in all_files if f.lower().endswith(".csv")]
print(len(images), "image files |", len(csv_files), "CSV files")

records = []
if csv_files:
    # CSV-driven layout (id + grade column per row), same column detection as the v1 notebook.
    by_stem, by_name = {}, {}
    for f in images:
        by_name.setdefault(os.path.basename(f), []).append(f)
        by_stem.setdefault(os.path.splitext(os.path.basename(f))[0], []).append(f)
    for csv_path in csv_files:
        sub = pd.read_csv(csv_path)
        base_dir = os.path.dirname(csv_path)
        id_col = next((c for c in sub.columns if any(k in c.lower() for k in ("id", "image", "file"))), None)
        label_col = next((c for c in sub.columns if any(k in c.lower() for k in ("diagnosis", "label", "grade", "level"))), None)
        if id_col is None or label_col is None:
            print(f"skipping {csv_path}: columns {list(sub.columns)}")
            continue
        for ident, grade in zip(sub[id_col].astype(str), sub[label_col]):
            candidates = by_name.get(ident) if ident.lower().endswith(IMG_EXTS) else by_stem.get(ident)
            candidates = sorted(c for c in (candidates or []) if c.startswith(base_dir))
            if candidates:
                records.append({"filepath": candidates[0], "diagnosis": int(grade)})
else:
    # Folder-per-grade layout: .../0/*.png ... .../4/*.png
    for f in images:
        grade = os.path.basename(os.path.dirname(f))
        if grade in {"0", "1", "2", "3", "4"}:
            records.append({"filepath": f, "diagnosis": int(grade)})

df = pd.DataFrame(records).drop_duplicates("filepath").reset_index(drop=True)
assert len(df) > 0, "No images matched: check the printed dataset path."
df["label"] = (df["diagnosis"] > 0).astype(int)   # 0 = no DR, 1 = DR present (ICDR 1-4), same as v1 and the app
print(len(df), "labelled images (v1 notebook had 236,170)")
print(df["diagnosis"].value_counts().sort_index().to_dict())

## 3. Split by image group: train 70% / validation 15% / test 15%

In [ ]:
def base_id(path):
    # Same rule as v1: strip augmentation suffixes so copies of one photo share a group.
    stem = os.path.splitext(os.path.basename(path))[0]
    return re.sub(r"(_aug\d*|_flip\w*|_rot(ate)?\d*|_crop\d*|_v\d+|_\d+)$", "", stem, flags=re.IGNORECASE)

df["group"] = df["filepath"].apply(base_id)
groups = df.groupby("group")["label"].max().reset_index()
train_g, rest_g = train_test_split(groups, test_size=0.30, stratify=groups["label"], random_state=SEED)
val_g, test_g = train_test_split(rest_g, test_size=0.50, stratify=rest_g["label"], random_state=SEED)

train_df = df[df["group"].isin(set(train_g["group"]))].reset_index(drop=True)
val_df = df[df["group"].isin(set(val_g["group"]))].reset_index(drop=True)
test_df = df[df["group"].isin(set(test_g["group"]))].reset_index(drop=True)

GROUP_OVERLAP = (set(train_df.group) & set(val_df.group)) | (set(train_df.group) & set(test_df.group)) | (set(val_df.group) & set(test_df.group))
print(f"groups: {len(groups)} for {len(df)} images (if these are equal, augmented copies could not be grouped)")
for name, part in (("train", train_df), ("validation", val_df), ("test", test_df)):
    print(f"{name:>10}: {len(part):7d} images | DR present {int(part.label.sum()):6d} | no DR {int((part.label == 0).sum()):6d}")
print("group overlap between splits:", len(GROUP_OVERLAP))

## 4. Input pipeline (bilinear, anti-aliased resize; MobileNetV2 scaling to [-1, 1])

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG), method="bilinear", antialias=True)
    return img / 127.5 - 1.0

def make_ds(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame["filepath"].values, frame["label"].values.astype("float32")))
    if training:
        ds = ds.shuffle(min(len(frame), 20000), seed=SEED, reshuffle_each_iteration=True).repeat()
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (tf.image.random_flip_left_right(x), y), num_parallel_calls=AUTOTUNE)
        ds = ds.apply(tf.data.experimental.ignore_errors())
    return ds.batch(BATCH).prefetch(AUTOTUNE)

monitor_df = val_df.sample(min(VAL_MONITOR_N, len(val_df)), random_state=SEED).reset_index(drop=True)
train_ds, monitor_ds = make_ds(train_df, training=True), make_ds(monitor_df)
steps = max(1, len(train_df) // BATCH) if STEPS_PER_EPOCH is None else min(STEPS_PER_EPOCH, max(1, len(train_df) // BATCH))
weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["label"].values)
CLASS_WEIGHT = {0: float(weights[0]), 1: float(weights[1])}
print("steps per epoch:", steps, "| class weights:", CLASS_WEIGHT)

## 5. Train: head first (backbone frozen), then fine-tune the top blocks

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

def build(weights="imagenet"):
    base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=weights)
    base.trainable = False
    inputs = layers.Input((IMG, IMG, 3))
    x = base(inputs, training=False)          # BatchNorm stays in inference mode while fine-tuning
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return Model(inputs, outputs), base

class TimeBudget(tf.keras.callbacks.Callback):
    def __init__(self, minutes):
        super().__init__()
        self.deadline = time.time() + minutes * 60
    def on_epoch_end(self, epoch, logs=None):
        if time.time() > self.deadline:
            print(f"time budget reached after epoch {epoch + 1}; stopping")
            self.model.stop_training = True

model, base = build("imagenet" if (not SMOKE or os.environ.get("DR_SMOKE_IMAGENET") == "1") else None)
best_path = os.path.join(OUT, "best.weights.h5")
checkpoint = tf.keras.callbacks.ModelCheckpoint(best_path, monitor="val_auc", mode="max",
                                                save_best_only=True, save_weights_only=True, verbose=1)
auc = tf.keras.metrics.AUC(name="auc")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=[auc])
h1 = model.fit(train_ds, steps_per_epoch=steps, epochs=HEAD_EPOCHS, validation_data=monitor_ds,
               class_weight=CLASS_WEIGHT, callbacks=[checkpoint], verbose=2)

base.trainable = True
unfreeze = False
for layer in base.layers:
    unfreeze = unfreeze or layer.name == FINE_TUNE_FROM
    layer.trainable = unfreeze and not isinstance(layer, layers.BatchNormalization)
print("trainable backbone layers:", sum(l.trainable for l in base.layers), "of", len(base.layers))

model.compile(optimizer=tf.keras.optimizers.Adam(2e-5), loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc")])
h2 = model.fit(train_ds, steps_per_epoch=steps, initial_epoch=HEAD_EPOCHS, epochs=HEAD_EPOCHS + FINE_EPOCHS,
               validation_data=monitor_ds, class_weight=CLASS_WEIGHT, verbose=2,
               callbacks=[checkpoint, TimeBudget(TIME_BUDGET_MIN),
                          tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=2)])

model.load_weights(best_path)   # the epoch with the best validation AUC
history = pd.concat([pd.DataFrame(h1.history), pd.DataFrame(h2.history)], ignore_index=True)
history.to_csv(os.path.join(OUT, "history.csv"), index_label="epoch")
TRAIN_MINUTES = (time.time() - T0) / 60
print(history.round(4))
print(f"elapsed so far: {TRAIN_MINUTES:.1f} min")

## 6. Threshold on validation, then the test split measured once

In [ ]:
def predict(frame):
    return model.predict(make_ds(frame), verbose=0).ravel().astype(float)

def metrics_at(y, p, thr):
    pred = p >= thr
    tp = int(np.sum(pred & (y == 1))); fn = int(np.sum(~pred & (y == 1)))
    tn = int(np.sum(~pred & (y == 0))); fp = int(np.sum(pred & (y == 0)))
    div = lambda a, b: float(a / b) if b else 0.0
    sens, spec, prec = div(tp, tp + fn), div(tn, tn + fp), div(tp, tp + fp)
    return {"threshold": float(thr), "sensitivity": sens, "specificity": spec, "precision": prec,
            "accuracy": div(tp + tn, len(y)), "f1": div(2 * prec * sens, prec + sens),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def select_threshold(y, p, target):
    # Highest threshold whose sensitivity >= target (same rule as backend/app/ml/thresholding.py).
    candidates = np.unique(p)[::-1]
    pos = p[y == 1]
    for thr in candidates:
        if np.mean(pos >= thr) >= target:
            return float(thr)
    return float(candidates[-1])

def bootstrap_ci(y, p, thr, key, n=BOOTSTRAP):
    rng = np.random.default_rng(SEED)
    vals = [metrics_at(y[idx], p[idx], thr)[key] for idx in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]

p_val, p_test = predict(val_df), predict(test_df)
y_val, y_test = val_df["label"].values, test_df["label"].values
THRESHOLD = select_threshold(y_val, p_val, TARGET_SENSITIVITY)

at_thr = metrics_at(y_test, p_test, THRESHOLD)
at_half = metrics_at(y_test, p_test, 0.5)
at_thr["sensitivity_95ci"] = bootstrap_ci(y_test, p_test, THRESHOLD, "sensitivity")
at_thr["specificity_95ci"] = bootstrap_ci(y_test, p_test, THRESHOLD, "specificity")
test_auc, val_auc = float(roc_auc_score(y_test, p_test)), float(roc_auc_score(y_val, p_val))

# Sensitivity per ICDR grade and for referable DR (grade >= 2) at the chosen threshold.
grades = test_df["diagnosis"].values
per_grade = {int(g): {"images": int(np.sum(grades == g)), "flagged": float(np.mean(p_test[grades == g] >= THRESHOLD))}
             for g in sorted(set(grades))}
ref = grades >= 2
referable = {"sensitivity": float(np.mean(p_test[ref] >= THRESHOLD)) if ref.any() else None,
             "specificity_vs_grade0": float(np.mean(p_test[grades == 0] < THRESHOLD)),
             "auc_grade0_vs_grade2plus": float(roc_auc_score(ref[(grades == 0) | ref], p_test[(grades == 0) | ref]))
             if ref.any() else None}

print(f"validation AUC {val_auc:.3f} | test AUC {test_auc:.3f}")
print(f"threshold chosen on validation for sensitivity >= {TARGET_SENSITIVITY}: {THRESHOLD:.4f}")
print("TEST at chosen threshold:", {k: round(v, 3) if isinstance(v, float) else v for k, v in at_thr.items()})
print("TEST at 0.5             :", {k: round(v, 3) if isinstance(v, float) else v for k, v in at_half.items()})
print("TEST share flagged per ICDR grade:", per_grade)
print("TEST referable DR (grade >= 2):", referable)
pd.DataFrame({"file": test_df["filepath"].map(os.path.basename), "diagnosis": grades, "label": y_test,
              "prob": p_test}).to_csv(os.path.join(OUT, "test_predictions.csv"), index=False)
pd.DataFrame({"file": val_df["filepath"].map(os.path.basename), "diagnosis": val_df["diagnosis"].values,
              "label": y_val, "prob": p_val}).to_csv(os.path.join(OUT, "val_predictions.csv"), index=False)

## 7. Export model.tflite (float32), check agreement, save parity samples for the app

In [ ]:
tf.keras.mixed_precision.set_global_policy("float32")
export_model, _ = build(None)
export_model.set_weights(model.get_weights())
export_model.save(os.path.join(OUT, "model.keras"))

try:
    tflite_bytes = tf.lite.TFLiteConverter.from_keras_model(export_model).convert()
except Exception as exc:  # some TF/Keras versions only convert from a SavedModel
    print("from_keras_model failed, using SavedModel export:", type(exc).__name__)
    saved_dir = os.path.join(OUT, "saved_model")
    export_model.export(saved_dir)
    tflite_bytes = tf.lite.TFLiteConverter.from_saved_model(saved_dir).convert()
tflite_path = os.path.join(OUT, "model.tflite")
open(tflite_path, "wb").write(tflite_bytes)
SHA256 = hashlib.sha256(tflite_bytes).hexdigest()

interp = tf.lite.Interpreter(model_path=tflite_path)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]

def tflite_score(x):
    interp.set_tensor(inp["index"], x[None].astype(np.float32))
    interp.invoke()
    return float(interp.get_tensor(out["index"]).ravel()[0])

agree_df = test_df.sample(min(256, len(test_df)), random_state=SEED).reset_index(drop=True)
x_agree = np.concatenate([xb.numpy() for xb, _ in make_ds(agree_df)])
keras_scores = export_model.predict(x_agree, verbose=0).ravel()
tfl_scores = np.array([tflite_score(x) for x in x_agree])
MAX_DIFF = float(np.max(np.abs(keras_scores - tfl_scores)))
print(f"tflite size {len(tflite_bytes) / 1e6:.1f} MB | sha256 {SHA256[:16]}... | max |keras - tflite| = {MAX_DIFF:.5f}")

# 24 test photos + their tflite scores, so the app's own preprocessing can be checked against training.
parity_dir = os.path.join(OUT, "parity")
os.makedirs(parity_dir, exist_ok=True)
pos = test_df[test_df.label == 1].sample(min(12, int(test_df.label.sum())), random_state=SEED)
neg = test_df[test_df.label == 0].sample(min(12, int((test_df.label == 0).sum())), random_state=SEED)
parity_rows = []
for i, row in enumerate(pd.concat([pos, neg]).itertuples()):
    name = f"sample_{i:02d}{os.path.splitext(row.filepath)[1].lower()}"
    shutil.copy(row.filepath, os.path.join(parity_dir, name))
    x = load_image(tf.constant(row.filepath)).numpy()
    parity_rows.append({"file": name, "diagnosis": int(row.diagnosis), "label": int(row.label), "tflite_score": tflite_score(x)})
pd.DataFrame(parity_rows).to_csv(os.path.join(parity_dir, "parity.csv"), index=False)

## 8. Save results and package everything

In [ ]:
results = {
    "model": "MobileNetV2 (ImageNet) fine-tuned from " + FINE_TUNE_FROM + "; head GAP-Dropout(0.3)-Dense(64)-Dropout(0.2)-Dense(1, sigmoid)",
    "dataset": "Kaggle ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy",
    "label": "binary: 0 = no DR (ICDR 0), 1 = DR present (ICDR 1-4)",
    "input": {"size": IMG, "resize": "bilinear, anti-aliased", "normalisation": "pixel / 127.5 - 1"},
    "split": {"method": "by image group (augmentation suffixes stripped), stratified, seed 42",
              "train": len(train_df), "validation": len(val_df), "test": len(test_df),
              "test_positives": int(y_test.sum()), "groups": int(len(groups)), "group_overlap": len(GROUP_OVERLAP)},
    "training": {"head_epochs": HEAD_EPOCHS, "max_fine_tune_epochs": FINE_EPOCHS, "steps_per_epoch": steps,
                 "batch": BATCH, "epochs_run": int(len(history)), "best_val_auc_monitor": float(history["val_auc"].max()),
                 "class_weight": CLASS_WEIGHT, "minutes": round(TRAIN_MINUTES, 1)},
    "threshold": {"value": THRESHOLD, "rule": f"highest validation threshold with sensitivity >= {TARGET_SENSITIVITY}",
                  "validation_at_threshold": metrics_at(y_val, p_val, THRESHOLD)},
    "test": {"auc": test_auc, "at_threshold": at_thr, "at_0.5": at_half, "per_grade_flagged": per_grade,
             "referable_dr": referable},
    "validation_auc": val_auc,
    "targets": {"sensitivity": TARGET_SENSITIVITY, "specificity": TARGET_SPECIFICITY,
                "sensitivity_met": at_thr["sensitivity"] >= TARGET_SENSITIVITY,
                "specificity_met": at_thr["specificity"] >= TARGET_SPECIFICITY},
    "export": {"tflite_sha256": SHA256, "tflite_bytes": len(tflite_bytes), "max_abs_keras_tflite_diff": MAX_DIFF},
    "environment": {"tensorflow": tf.__version__, "gpus": len(GPUS), "smoke_test": SMOKE},
}
json.dump(results, open(os.path.join(OUT, "results.json"), "w"), indent=2)

zip_path = os.path.join(OUT, "dr_v2_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("model.tflite", "model.keras", "results.json", "history.csv", "test_predictions.csv", "val_predictions.csv"):
        z.write(os.path.join(OUT, name), name)
    for name in sorted(os.listdir(parity_dir)):
        z.write(os.path.join(parity_dir, name), os.path.join("parity", name))
print("saved", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB) | total {(time.time() - T0) / 60:.1f} min")

## 9. CHECKS (screenshot this cell)

In [ ]:
checks = [
    ("no image group appears in two splits", len(GROUP_OVERLAP) == 0),
    ("test split has at least 1,000 DR-present images", int(y_test.sum()) >= (10 if SMOKE else 1000)),
    ("model learned something (test AUC > 0.70)", test_auc > (0.0 if SMOKE else 0.70)),
    ("tflite matches keras (max difference < 0.01)", MAX_DIFF < 0.01),
    ("threshold chosen on validation only", True),
]
for name, ok in checks:
    print(("PASS  " if ok else "FAIL  ") + name)
print()
print(f"Test sensitivity {at_thr['sensitivity']:.3f} (95% CI {at_thr['sensitivity_95ci'][0]:.3f}-{at_thr['sensitivity_95ci'][1]:.3f}), "
      f"specificity {at_thr['specificity']:.3f} (95% CI {at_thr['specificity_95ci'][0]:.3f}-{at_thr['specificity_95ci'][1]:.3f}), "
      f"AUC {test_auc:.3f}, threshold {THRESHOLD:.4f}")
print("Proposal targets: sensitivity >= 0.85", "MET" if results["targets"]["sensitivity_met"] else "NOT MET",
      "| specificity >= 0.80", "MET" if results["targets"]["specificity_met"] else "NOT MET")
print("VERDICT:", "results can be used" if all(ok for _, ok in checks) else "do NOT use these results; send the FAIL lines to Claude Code")